# LECTURE 17 — HANDS-ON EXERCISE
### MODULE 7: UNSUPERVISED LEARNING & DIMENSIONALITY REDUCTION

**HANDS-ON EXERCISE** — Develop Peer-Group Classifications for African Economies

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: FINANCIAL DEVELOPMENT DATA (provided)

In [ ]:
COUNTRIES = [
    'Nigeria',        'South Africa',   'Egypt',          'Kenya',
    'Ethiopia',       'Ghana',          'Tanzania',       "Cote d'Ivoire",
    'Cameroon',       'Angola',         'Uganda',         'Mozambique',
    'Zambia',         'Zimbabwe',       'Senegal',        'Mali',
    'Burkina Faso',   'Guinea',         'Niger',          'Rwanda',
    'Benin',          'Burundi',        'Sierra Leone',   'Togo',
    'Libya',          'Tunisia',        'Algeria',        'Morocco',
    'Sudan',          'Madagascar',     'Botswana',       'Namibia',
    'Mauritius',      'Malawi',         'DR Congo',       'Congo Republic',
    'Gabon',          'Eq. Guinea',     'Eswatini',       'Lesotho',
    'Djibouti',       'Eritrea',        'Somalia',        'Chad',
    'Cabo Verde',
]


def generate_fin_dev_data(countries: list, seed: int = 77) -> pd.DataFrame:
    """
    Synthetic financial development & inclusion indicators for 45 economies.

    Indicators:
      bank_branches_per_100k  — commercial bank branches per 100,000 adults
      mobile_money_adoption   — % adults with mobile money account
      private_credit_gdp      — private sector credit (% of GDP)
      insurance_penetration   — insurance premiums (% of GDP)
      stock_market_cap_gdp    — stock market capitalisation (% of GDP)
      remittance_inflows_gdp  — remittances received (% of GDP)
      financial_inclusion_idx — composite index (0–100)
    """
    rng = np.random.default_rng(seed)

    # Latent groups: Developed (0), Emerging (1), Frontier (2)
    groups = np.array([0]*13 + [1]*17 + [2]*15)

    centres = np.array([
        [18,  45,  55,  4.5,  80,  3.0,  70],   # Developed
        [ 8,  35,  28,  1.8,  20,  6.5,  45],   # Emerging
        [ 3,  20,  12,  0.6,   5,  9.0,  22],   # Frontier
    ], dtype=float)

    noise = np.array([3, 8, 8, 0.6, 12, 1.5, 8], dtype=float)

    X = np.zeros((len(countries), 7))
    for i, g in enumerate(groups):
        X[i] = centres[g] + rng.normal(0, noise)

    X[:, 0] = np.clip(X[:, 0],  0.5,  35)
    X[:, 1] = np.clip(X[:, 1],  2,    75)
    X[:, 2] = np.clip(X[:, 2],  3,    80)
    X[:, 3] = np.clip(X[:, 3],  0.1,   7)
    X[:, 4] = np.clip(X[:, 4],  0,   120)
    X[:, 5] = np.clip(X[:, 5],  0.5,  18)
    X[:, 6] = np.clip(X[:, 6],  5,   100)

    cols = ['bank_branches_per_100k', 'mobile_money_adoption',
            'private_credit_gdp',     'insurance_penetration',
            'stock_market_cap_gdp',   'remittance_inflows_gdp',
            'financial_inclusion_idx']

    return pd.DataFrame(X, index=countries, columns=cols)


df = generate_fin_dev_data(COUNTRIES)
INDICATOR_COLS = df.columns.tolist()

print(f"Dataset : {df.shape[0]} African economies  x  {len(INDICATOR_COLS)} indicators")
print(df.describe().round(2).to_string())

# Quick radar preview (provided)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Financial Development Indicators — Distribution across African Economies',
             fontsize=12, fontweight='bold')
for ax, col in zip(axes.flatten(), INDICATOR_COLS):
    ax.hist(df[col], bins=20, color='steelblue', alpha=0.75, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontsize=9)
    ax.grid(True, alpha=0.3)
axes.flatten()[-1].set_visible(False)
plt.tight_layout()
plt.savefig('lecture17_exercise_distributions.png', bbox_inches='tight')
plt.show()
print("Saved : lecture17_exercise_distributions.png")

## ── SECTION 3: STANDARDISE FEATURES (provided)

In [ ]:
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df[INDICATOR_COLS].values)
print("\nFeatures standardised.")

## ── SECTION 4: DETERMINE OPTIMAL k  ◄─ YOUR TASK (TODO 1)

Run K-Means for k = 2 to 10.

For each k, compute:

(a) inertia      : km.inertia_

(b) silhouette   : silhouette_score(X_sc, km.labels_)

Store results and plot both metrics side by side.

Identify the optimal k (highest silhouette score) and print it.

Hint: KMeans(n_clusters=k, random_state=SEED, n_init=10)

In [ ]:
# WRITE YOUR CODE BELOW
# K_RANGE    = range(2, 11)
# inertias   = []
# sil_scores = []
#
# for k in K_RANGE:
#     ...
#
# # Plot elbow and silhouette
# ...
# optimal_k = ...
# print(f"Optimal k : {optimal_k}")

## ── SECTION 5: APPLY K-MEANS  ◄─ YOUR TASK (TODO 2)

Using K_FINAL = 3 (or your optimal k from TODO 1):

1. Fit KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=20)

2. Assign cluster labels to df['kmeans_cluster']

3. Print the cluster profile table:

df.groupby('kmeans_cluster')[INDICATOR_COLS].mean().round(2)

In [ ]:
# WRITE YOUR CODE BELOW
K_FINAL = 3   # you may change this based on your TODO 1 result

# kmeans = KMeans(...)
# df['kmeans_cluster'] = kmeans.fit_predict(X_sc)
# print(df.groupby('kmeans_cluster')[INDICATOR_COLS].mean().round(2))

## ── SECTION 6: HIERARCHICAL CLUSTERING  ◄─ YOUR TASK (TODO 3)

1. Compute linkage matrix using Ward linkage:

Z = linkage(X_sc, method='ward')

2. Plot the dendrogram:

dendrogram(Z, labels=df.index.tolist(), leaf_rotation=90, leaf_font_size=7)

3. Assign hierarchical cluster labels:

agg = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')

df['hier_cluster'] = agg.fit_predict(X_sc)

4. Print the cluster membership (which countries are in each group).

In [ ]:
# WRITE YOUR CODE BELOW
# Z = linkage(X_sc, method='ward')
# ...

## ── SECTION 7: VISUALISE CLUSTERS (PCA SCATTER)  ◄─ YOUR TASK (TODO 4)

Project the 7-dimensional data to 2D using PCA and create a scatter plot

coloured by K-Means cluster labels.

Steps:

pca2   = PCA(n_components=2, random_state=SEED)

coords = pca2.fit_transform(X_sc)

Plot coords[:, 0] vs coords[:, 1], colour each point by df['kmeans_cluster'].

Label each point with the country name (fontsize 6 is fine).

Title the axes with the variance explained:

f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f} % variance)"

In [ ]:
# WRITE YOUR CODE BELOW
# pca2   = PCA(n_components=2, random_state=SEED)
# coords = pca2.fit_transform(X_sc)
# ...

## ── SECTION 8: DISCUSSION QUESTIONS

In [ ]:
print("""
── Discussion Questions ───────────────────────────────────────────────────────
 1. How did your clusters change when you used financial development indicators
    compared to the vulnerability indicators from the demo?

 2. Which indicator most sharply separates the clusters?
    (Hint: look at the cluster profile table from TODO 2.)

 3. A central bank governor asks: "Should we benchmark against our K-Means
    peer group or our hierarchical peer group?" What would you advise?

 4. Mobile money adoption is high in some low-income economies. Does this
    create a surprising cluster membership? How would you handle it?

 5. Name one policy recommendation you would make for each peer group based
    on the financial development profiles.
──────────────────────────────────────────────────────────────────────────────
""")

print("=== EXERCISE COMPLETE ===")